# 6. GraphRAG Global Sensemaking over 183 Hospital EHR Tickets — and its Indexing Price

**Presentation map:** Slide 33 (Microsoft GraphRAG: build a graph, cluster it, summarise every community),
Slide 34 (GraphRAG wins on global sensemaking — at an indexing price), Slide 36 (graphs help on summarisation, not
on fact lookup), Slide 38 (decision rule: corpus-wide sensemaking → community summaries)

**Dataset:** `data/planetcare_tickets.json` — 183 generated (synthetic) PlanetCare EHR support tickets from 10 fictional hospitals, 28 classifications,
62 free-text module names, English and French, 122 open / 61 closed.

**The question type:** *"What are the main recurring problem themes across all hospitals?"* No single ticket holds that
answer, so top-k retrieval can only ever see a slice of the corpus. GraphRAG pays an up-front **indexing** cost
(LLM extraction + clustering + one LLM report per community) so that a global question can be answered by
**map-reduce over community reports** that together cover the whole corpus.

This is a compact re-implementation of the GraphRAG pipeline (not the `graphrag` package): batched LLM extraction →
graph → Louvain communities (Leiden in the original) → structured community reports → global map-reduce search —
evaluated against vector RAG with an LLM pairwise judge in **both orderings**, plus deterministic grounding checks
and a cost ledger.

In [1]:
import json, time, re
from collections import Counter, defaultdict
import networkx as nx

with open("data/planetcare_tickets.json") as f:
    raw = json.load(f)

# Two ticket ids occur twice in the export; keep both records under distinct node ids
seen, tickets = Counter(), []
for t in raw:
    seen[t["ticket_id"]] += 1
    tickets.append({**t, "tid": t["ticket_id"] if seen[t["ticket_id"]] == 1 else f"{t['ticket_id']}-b"})
TID = {t["tid"]: t for t in tickets}

print(f"{len(tickets)} tickets | {len({t['hospital'] for t in tickets})} hospitals | "
      f"{len({t['classification'] for t in tickets})} classifications | {len({t['module'] for t in tickets})} module names")
print("Status:", dict(Counter(t["status"] for t in tickets)))
print("Top classifications:", Counter(t["classification"] for t in tickets).most_common(6))

def ticket_text(t):
    res = t["resolution"] if t["resolution"] not in ("", "None") else "none"
    return (f"[{t['tid']}] {t['summary']}: {t['description']} | class={t['classification']} | hospital={t['hospital']} "
            f"| status={t['status']} | priority={t['priority']} | resolution={res}")
print("\nExample:", ticket_text(tickets[0]))

183 tickets | 10 hospitals | 28 classifications | 62 module names
Status: {'Open': 122, 'Closed': 61}
Top classifications: [('BILLING', 24), ('INTERFACING', 20), ('SYSTEM', 18), ('PATIENT_ADMIN', 10), ('WAITING_LISTS', 9), ('LABORATORY', 8)]

Example: [PC-00105] Issue with Questionnaire Submission: The questionnaires submitted by patients are not reflecting in the system for HN-0001. | class=QUESTIONNAIRES | hospital=Sunrise Medical Center | status=Open | priority=High | resolution=none


## A cost ledger — the "indexing price" is measured, not quoted

Every LLM and embedding call goes through one wrapper that records calls, tokens and wall-clock seconds per phase:
`index` (extraction, embeddings, community reports) versus `query:<method>`.

In [2]:
import os
import numpy as np
from openai import OpenAI
from pydantic import BaseModel, Field
from concurrent.futures import ThreadPoolExecutor

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
EMBED_MODEL, CHAT_MODEL, SEED = "text-embedding-3-small", "gpt-4o-mini", 7
ledger = defaultdict(lambda: {"calls": 0, "in_tok": 0, "out_tok": 0, "sec": 0.0})

def track(phase, usage, sec):
    L = ledger[phase]; L["calls"] += 1; L["sec"] += sec
    L["in_tok"] += usage.prompt_tokens; L["out_tok"] += getattr(usage, "completion_tokens", 0) or 0

def llm(phase, messages, schema=None):
    t0 = time.perf_counter()
    kw = dict(model=CHAT_MODEL, messages=messages, temperature=0, seed=SEED)
    r = client.chat.completions.parse(response_format=schema, **kw) if schema else client.chat.completions.create(**kw)
    track(phase, r.usage, time.perf_counter() - t0)
    return r.choices[0].message.parsed if schema else r.choices[0].message.content

def embed(phase, texts):
    t0 = time.perf_counter()
    r = client.embeddings.create(model=EMBED_MODEL, input=texts)
    track(phase, r.usage, time.perf_counter() - t0)
    v = np.array([e.embedding for e in r.data]); return v / np.linalg.norm(v, axis=1, keepdims=True)

def pmap(fn, items, workers=8):
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(fn, items))

## Indexing step 1 — LLM extraction into a graph (Slide 33: "LLM extracts entities + relations")

Tickets are sent in batches of 20 to a structured-output extractor that returns, per ticket, a canonical **issue**
(a reusable name, in English, even for French tickets) and the affected **component**. The graph gets
`(Ticket)-[:REPORTED_AT]->(Hospital)`, `(Ticket)-[:CLASSIFIED_AS]->(Classification)`,
`(Ticket)-[:EXHIBITS]->(Issue)` and `(Issue)-[:AFFECTS]->(Component)`.

In [3]:
class Extraction(BaseModel):
    tid: str
    issue: str = Field(description="Canonical reusable issue name in English, 2-5 words, e.g. 'Lab results not displayed'")
    component: str = Field(description="Affected system component in English, e.g. 'Lab interface', 'Billing engine'")

class Batch(BaseModel):
    items: list[Extraction]

def extract(batch):
    out = llm("index", [
        {"role": "system", "content": "You turn raw EHR support tickets into canonical graph entities. Reuse the same "
         "issue name for tickets that describe the same underlying problem."},
        {"role": "user", "content": "Return one item per ticket:\n" + "\n".join(ticket_text(t) for t in batch)}], Batch)
    return out.items

t_index = time.perf_counter()
batches = [tickets[i:i + 20] for i in range(0, len(tickets), 20)]
extracted = {e.tid: e for b in pmap(extract, batches) for e in b if e.tid in TID}
print(f"Extracted {len(extracted)}/{len(tickets)} tickets in {len(batches)} batched calls")

norm = lambda s: re.sub(r"[^a-z0-9 ]", "", s.lower()).strip()
G = nx.Graph()
for t in tickets:
    G.add_node(t["tid"], label="Ticket")
    for lab, val, rel in (("Hospital", t["hospital"], "REPORTED_AT"), ("Classification", t["classification"], "CLASSIFIED_AS")):
        G.add_node((lab, val), label=lab); G.add_edge(t["tid"], (lab, val), type=rel)
    if (e := extracted.get(t["tid"])):
        iss, comp = ("Issue", norm(e.issue)), ("Component", norm(e.component))
        G.add_node(iss, label="Issue", name=e.issue); G.add_node(comp, label="Component", name=e.component)
        G.add_edge(t["tid"], iss, type="EXHIBITS"); G.add_edge(iss, comp, type="AFFECTS")
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges | "
      f"{dict(Counter(d['label'] for _, d in G.nodes(data=True)))}")

Extracted 180/183 tickets in 10 batched calls
Graph: 429 nodes, 702 edges | {'Ticket': 183, 'Hospital': 10, 'Classification': 28, 'Issue': 145, 'Component': 63}


## Indexing step 2 — communities (Slide 33: "Hierarchical Leiden → communities")

GraphRAG clusters the extracted graph so that each community is a coherent slice of the corpus. We project onto
tickets: two tickets are linked when they share an extracted **issue** or **component** or a **classification**, and
when their embeddings are mutual near neighbours (cosine kNN, k = 5). Louvain (fixed seed) then partitions this ticket
graph; hospitals are deliberately *not* used for linking, so communities are about problems, not sites.

In [4]:
ids = [t["tid"] for t in tickets]
V = embed("index", [ticket_text(TID[i]) for i in ids])
S = V @ V.T

P = nx.Graph(); P.add_nodes_from(ids)
def link(a, b, w):
    P.add_edge(a, b, weight=P[a][b]["weight"] + w if P.has_edge(a, b) else w)
for lab, w in (("Issue", 1.0), ("Component", 0.5), ("Classification", 0.5)):
    for n, d in G.nodes(data=True):
        if d["label"] == lab:
            members = [m for m in G.neighbors(n) if m in TID] if lab != "Component" else \
                      [m for iss in G.neighbors(n) if G.nodes[iss]["label"] == "Issue" for m in G.neighbors(iss) if m in TID]
            for i in range(len(members)):
                for j in range(i + 1, len(members)):
                    link(members[i], members[j], w)
knn = np.argsort(-S, axis=1)[:, 1:6]
for i, row in enumerate(knn):
    for j in row:
        if i in knn[j]:                                   # mutual nearest neighbours only
            link(ids[i], ids[j], float(S[i, j]))

comms = sorted(nx.community.louvain_communities(P, weight="weight", resolution=1.0, seed=SEED), key=len, reverse=True)
communities = [sorted(c) for c in comms if len(c) >= 3]
leftover = sorted(t for c in comms if len(c) < 3 for t in c)
if leftover:
    communities.append(leftover)
index_seconds_graph = time.perf_counter() - t_index
print(f"Ticket graph: {P.number_of_edges()} weighted edges -> {len(communities)} communities "
      f"(sizes {[len(c) for c in communities]})")
for k, c in enumerate(communities):
    cls = Counter(TID[t]["classification"] for t in c).most_common(3)
    print(f"  C{k:<2} {len(c):>3} tickets | {len({TID[t]['hospital'] for t in c})} hospitals | top classes {cls}")

Ticket graph: 965 weighted edges -> 18 communities (sizes [27, 25, 20, 15, 13, 12, 11, 9, 7, 6, 6, 6, 6, 6, 4, 4, 3, 3])
  C0   27 tickets | 10 hospitals | top classes [('SYSTEM', 18), ('PERFORMANCE', 4), ('CONFIGURATION', 3)]
  C1   25 tickets | 7 hospitals | top classes [('BILLING', 24), ('REVENUE_CYCLE', 1)]
  C2   20 tickets | 10 hospitals | top classes [('INTERFACING', 20)]
  C3   15 tickets | 8 hospitals | top classes [('USER_MAINTENANCE', 5), ('CLINICAL_NOTES', 4), ('CLINICAL_WORKBENCHES', 3)]
  C4   13 tickets | 8 hospitals | top classes [('ELECTRONIC_PATIENT_RECORD', 5), ('MATERNITY', 4), ('MEDICAL_RECORDS', 4)]
  C5   12 tickets | 9 hospitals | top classes [('LABORATORY', 8), ('OPERATING_THEATRES', 4)]
  C6   11 tickets | 7 hospitals | top classes [('PATIENT_ADMIN', 10), ('CONFIGURATION', 1)]
  C7    9 tickets | 3 hospitals | top classes [('WAITING_LISTS', 9)]
  C8    7 tickets | 6 hospitals | top classes [('ORDERS', 7)]
  C9    6 tickets | 5 hospitals | top classes [('PRINTI

## Indexing step 3 — one structured community report per community (Slide 33: "LLM community reports (JSON)")

Each report names the theme, summarises it, lists affected hospitals, counts open tickets, says whether a KB article
is needed, and **cites the ticket ids** it is based on — exactly the role of GraphRAG community reports, which cite
their evidence as entity and relationship ids.

In [5]:
class CommunityReport(BaseModel):
    title: str = Field(description="Name of the theme itself, e.g. 'Lab results not reaching the EMR' (not 'Community report')")
    summary: str = Field(description="3-5 sentences on the recurring problem(s) and their operational impact")
    key_issues: list[str]
    affected_hospitals: list[str]
    open_tickets: int
    kb_article_needed: bool
    suggested_kb_title: str
    cited_ticket_ids: list[str] = Field(description="ids of the tickets that best support the summary")

def report(k_c):
    k, c = k_c
    issues = Counter(extracted[t].issue for t in c if t in extracted).most_common(8)
    r = llm("index", [
        {"role": "system", "content": "You write GraphRAG community reports for an EHR support team. Use only the tickets given."},
        {"role": "user", "content": f"Community C{k}: {len(c)} tickets, {sum(TID[t]['status'] == 'Open' for t in c)} open. "
         f"Most frequent extracted issues: {issues}\n\nTickets:\n" + "\n".join(ticket_text(TID[t]) for t in c)}], CommunityReport)
    return {"id": f"C{k}", "size": len(c), "members": c, **r.model_dump()}

reports = pmap(report, list(enumerate(communities)))
index_seconds = time.perf_counter() - t_index
for r in reports:
    print(f"{r['id']:<4} ({r['size']:>3} tickets, {r['open_tickets']:>2} open, KB needed={r['kb_article_needed']!s:<5}) {r['title']}")

C0   ( 27 tickets, 17 open, KB needed=True ) System Performance and Outage Issues
C1   ( 25 tickets, 15 open, KB needed=True ) Billing Discrepancies and Inquiries
C2   ( 20 tickets, 16 open, KB needed=True ) Interfacing Issues Across Multiple Hospitals
C3   ( 15 tickets, 10 open, KB needed=True ) User Maintenance and Clinical Notes Issues
C4   ( 13 tickets, 11 open, KB needed=True ) Electronic Patient Record Access Issues
C5   ( 12 tickets,  8 open, KB needed=True ) Laboratory Results and Operating Theatre Issues
C6   ( 11 tickets,  5 open, KB needed=True ) Patient Administration Issues Across Multiple Hospitals
C7   (  9 tickets,  7 open, KB needed=True ) Waiting List Management Issues
C8   (  7 tickets,  5 open, KB needed=True ) Order Processing and Fulfillment Issues
C9   (  6 tickets,  3 open, KB needed=True ) Recurring Printing Issues Across Multiple Hospitals
C10  (  6 tickets,  6 open, KB needed=True ) Questionnaire Submission Issues
C11  (  6 tickets,  3 open, KB needed=True ) 

## Query time — global map-reduce vs. vector RAG (Slide 33: "Global: map-reduce over community reports")

* **GraphRAG global search** — *map*: each community report produces a partial answer with a 0-100 helpfulness
  score; *reduce*: the helpful partial answers (score > 0) are merged into one final answer that cites ticket ids.
* **Vector RAG** — embed the question, retrieve the top-15 most similar tickets, answer from them (citing ids).

In [6]:
QUESTIONS = [
    "What are the main recurring problem themes across PlanetCare hospitals, and which of them most need a new KB article?",
    "Which problems are spread across the most hospitals, and are they mostly resolved or still open?",
    "Where are the biggest risks to patient care and hospital operations in the current PlanetCare support backlog?",
]
CITE = "Cite supporting ticket ids in square brackets, e.g. [PC-00013]."

class Partial(BaseModel):
    helpfulness: int = Field(ge=0, le=100)
    points: str

def graphrag_global(question):
    def map_one(r):
        rep = {k: r[k] for k in ("id", "size", "title", "summary", "key_issues", "affected_hospitals",
                                 "open_tickets", "kb_article_needed", "suggested_kb_title", "cited_ticket_ids")}
        return llm("query:graphrag", [{"role": "user", "content":
            f"Question: {question}\n\nCommunity report:\n{json.dumps(rep)}\n\nGive the points from this report that help "
            f"answer the question ({CITE}) and rate helpfulness 0-100 (0 = irrelevant)."}], Partial)
    partials = sorted(zip(pmap(map_one, reports), reports), key=lambda x: -x[0].helpfulness)
    kept = [f"(from {r['id']}, {r['size']} tickets, score {p.helpfulness}) {p.points}" for p, r in partials if p.helpfulness > 0]
    return llm("query:graphrag", [{"role": "user", "content":
        f"Question: {question}\n\nPartial answers from community reports covering all {len(tickets)} tickets:\n"
        + "\n\n".join(kept) + f"\n\nWrite the final answer (max ~350 words). {CITE}"}])

def vector_rag(question, k=15):
    q = embed("query:vector", [question])[0]
    top = np.argsort(-(V @ q))[:k]
    return llm("query:vector", [{"role": "user", "content":
        f"Question: {question}\n\nSupport tickets:\n" + "\n".join(ticket_text(TID[ids[i]]) for i in top)
        + f"\n\nAnswer using only these tickets (max ~350 words). {CITE}"}])

answers = {}
for q in QUESTIONS:
    answers[q] = {"graphrag": graphrag_global(q), "vector": vector_rag(q)}
q0 = QUESTIONS[0]
print("Q:", q0, "\n\n--- GraphRAG global ---\n" + answers[q0]["graphrag"] + "\n\n--- Vector RAG (top-15) ---\n" + answers[q0]["vector"])

Q: What are the main recurring problem themes across PlanetCare hospitals, and which of them most need a new KB article? 

--- GraphRAG global ---
The main recurring problem themes across PlanetCare hospitals include:

1. **System Performance Issues and Outages**: Significant performance issues, system crashes, and login errors are prevalent, particularly at Sunrise Medical Center and Valley Health System. There are 17 open tickets indicating the urgency of these problems. A new KB article titled 'Troubleshooting System Performance and Outages' is needed to address these issues [PC-00043, PC-00044, PC-00046, PC-00047, PC-00051, PC-00139, PC-00140, PC-00180, PC-00181].

2. **Billing Discrepancies and Inquiries**: This theme includes a high volume of unresolved billing issues affecting multiple hospitals, with 15 open tickets. A new KB article titled 'Best Practices for Addressing Billing Discrepancies and Inquiries' is necessary to improve processes [PC-00001, PC-00002, PC-00004, PC-000

## Evaluation — LLM pairwise judge, both orderings, plus deterministic grounding

Following the GraphRAG paper (Slide 34), a judge compares the two answers on **comprehensiveness** and **diversity**;
**directness** is included as a control criterion on which the GraphRAG paper found naive RAG wins. Each pair is judged
**twice with the answers swapped** to cancel position bias. Grounding is checked without an LLM: how many distinct
*real* ticket ids each answer cites, how many cited ids do not exist, and how many classifications and hospitals the
cited tickets span.

In [7]:
CRITERIA = {
    "comprehensiveness": "How much detail does the answer provide to cover all aspects and details of the question?",
    "diversity": "How varied and rich is the answer in providing different perspectives and insights on the question?",
    "directness": "How specifically and clearly does the answer address the question (concise, to the point)?",
}
class Judgement(BaseModel):
    winner: str = Field(description="'1', '2' or 'tie'")
    reason: str

def judge(args):
    q, crit, first = args
    a1, a2 = (answers[q][first], answers[q]["vector" if first == "graphrag" else "graphrag"])
    j = llm("judge", [{"role": "system", "content": "You are an impartial evaluator. Judge only the stated criterion."},
                      {"role": "user", "content": f"Question: {q}\nCriterion — {crit}: {CRITERIA[crit]}\n\n"
                       f"Answer 1:\n{a1}\n\nAnswer 2:\n{a2}\n\nWhich answer is better on this criterion?"}], Judgement)
    w = j.winner.strip().strip("'\"")
    return crit, ("tie" if w not in ("1", "2") else (first if w == "1" else ("vector" if first == "graphrag" else "graphrag")))

jobs = [(q, c, first) for q in QUESTIONS for c in CRITERIA for first in ("graphrag", "vector")]
verdicts = pmap(judge, jobs)
win = {}
for c in CRITERIA:
    vs = [w for cc, w in verdicts if cc == c]
    win[c] = (sum(w == "graphrag" for w in vs) + 0.5 * sum(w == "tie" for w in vs)) / len(vs)
    print(f"{c:<18} GraphRAG win rate {win[c]:.0%}  ({Counter(vs)})")

def grounding(ans):
    cited = set(re.findall(r"PC-\d{5}(?:-b)?", ans))
    real = [c for c in cited if c in TID]
    return {"cited": len(real), "invalid": len(cited) - len(real),
            "classes": len({TID[c]["classification"] for c in real}), "hospitals": len({TID[c]["hospital"] for c in real})}
gstats = {m: [grounding(answers[q][m]) for q in QUESTIONS] for m in ("graphrag", "vector")}
for m, gs in gstats.items():
    avg = {k: sum(g[k] for g in gs) / len(gs) for k in gs[0]}
    print(f"{m:<9} avg per answer: {avg['cited']:.1f} real ticket ids cited, {avg['invalid']:.1f} invalid, "
          f"{avg['classes']:.1f} classifications, {avg['hospitals']:.1f} hospitals spanned")

comprehensiveness  GraphRAG win rate 100%  (Counter({'graphrag': 6}))
diversity          GraphRAG win rate 83%  (Counter({'graphrag': 5, 'vector': 1}))
directness         GraphRAG win rate 100%  (Counter({'graphrag': 6}))
graphrag  avg per answer: 43.3 real ticket ids cited, 0.0 invalid, 8.7 classifications, 10.0 hospitals spanned
vector    avg per answer: 8.7 real ticket ids cited, 0.0 invalid, 5.0 classifications, 5.0 hospitals spanned


In [8]:
def phase_tot(prefix):
    ks = [k for k in ledger if k.startswith(prefix)]
    return {f: sum(ledger[k][f] for k in ks) for f in ("calls", "in_tok", "out_tok")}
idx, qg, qv = phase_tot("index"), phase_tot("query:graphrag"), phase_tot("query:vector")
nq = len(QUESTIONS)
print(f"{'Phase':<28} {'LLM/embed calls':>15} {'input tok':>10} {'output tok':>10}")
print(f"{'GraphRAG indexing (once)':<28} {idx['calls']:>15} {idx['in_tok']:>10,} {idx['out_tok']:>10,}   ({index_seconds:.0f} s wall clock, 8 parallel workers)")
print(f"{'GraphRAG query (per q)':<28} {qg['calls']/nq:>15.1f} {qg['in_tok']/nq:>10,.0f} {qg['out_tok']/nq:>10,.0f}")
print(f"{'Vector RAG indexing (once)':<28} {1:>15} {'(183 embeddings only)':>21}")
print(f"{'Vector RAG query (per q)':<28} {qv['calls']/nq:>15.1f} {qv['in_tok']/nq:>10,.0f} {qv['out_tok']/nq:>10,.0f}")

gavg = {m: {k: sum(g[k] for g in gs) / len(gs) for k in gs[0]} for m, gs in gstats.items()}
print("\n" + "=" * 74 + "\nRESULTS SUMMARY — Notebook 6 (GraphRAG global sensemaking, 183 PlanetCare tickets)\n" + "=" * 74)
print(f"  Index: {len(batches)} extraction calls -> graph {G.number_of_nodes()} nodes / {G.number_of_edges()} edges -> "
      f"{len(communities)} communities -> {len(reports)} reports")
for c in CRITERIA:
    print(f"  GraphRAG pairwise win rate, {c:<18} {win[c]:.0%}  ({len(QUESTIONS)} questions x 2 orderings)")
print(f"  Ticket ids cited per answer:   GraphRAG {gavg['graphrag']['cited']:.1f} | vector {gavg['vector']['cited']:.1f}"
      f"   (invalid: {gavg['graphrag']['invalid']:.1f} | {gavg['vector']['invalid']:.1f})")
print(f"  Classifications spanned:       GraphRAG {gavg['graphrag']['classes']:.1f} | vector {gavg['vector']['classes']:.1f}  (of 28)")
print(f"  Hospitals spanned:             GraphRAG {gavg['graphrag']['hospitals']:.1f} | vector {gavg['vector']['hospitals']:.1f}  (of 10)")
print(f"  Indexing price: GraphRAG {idx['calls']} calls / {idx['in_tok'] + idx['out_tok']:,} tokens vs vector 1 embedding call")
print(f"  Query price:    GraphRAG {qg['calls']/nq:.0f} calls / {(qg['in_tok'] + qg['out_tok'])/nq:,.0f} tok per question vs "
      f"vector {qv['calls']/nq:.0f} / {(qv['in_tok'] + qv['out_tok'])/nq:,.0f}")
print("\nLecture message (Slides 33/34/38): for corpus-wide questions, community summaries see the whole corpus while")
print("top-k retrieval sees a slice — but the graph has to be paid for up front, and global search is costlier")
print("per query. Use it when sensemaking is the main workload; otherwise prefer a cheaper index.")

Phase                        LLM/embed calls  input tok output tok
GraphRAG indexing (once)                  29     37,130      6,982   (13 s wall clock, 8 parallel workers)
GraphRAG query (per q)                  19.0      9,380      3,336
Vector RAG indexing (once)                 1 (183 embeddings only)
Vector RAG query (per q)                 2.0        839        365

RESULTS SUMMARY — Notebook 6 (GraphRAG global sensemaking, 183 PlanetCare tickets)
  Index: 10 extraction calls -> graph 429 nodes / 702 edges -> 18 communities -> 18 reports
  GraphRAG pairwise win rate, comprehensiveness  100%  (3 questions x 2 orderings)
  GraphRAG pairwise win rate, diversity          83%  (3 questions x 2 orderings)
  GraphRAG pairwise win rate, directness         100%  (3 questions x 2 orderings)
  Ticket ids cited per answer:   GraphRAG 43.3 | vector 8.7   (invalid: 0.0 | 0.0)
  Classifications spanned:       GraphRAG 8.7 | vector 5.0  (of 28)
  Hospitals spanned:             GraphRAG 10.0 | v